In [16]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

class AudioDataset(Dataset):
    def __init__(self, tensors, labels):
        self.tensors = tensors
        self.labels = labels

    def __len__(self):
        return len(self.tensors)

    def __getitem__(self, idx):
        return self.tensors[idx], self.labels[idx]

# Veriyi yükleme
tensors = torch.load('audio_tensors.pt')
df = pd.read_csv("processed.tsv", sep="\t")
labels = df["speaker_id"].values
dataset = AudioDataset(tensors, labels)
dataloader = DataLoader(dataset, batch_size=13, shuffle=True)

/tmp/ipykernel_6005/494413587.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tensors = torch.load('audio_tensors.pt')


In [17]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

# Modeli cihazda çalıştırma
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(input_size=29, hidden_size=128, output_size=64).to(device)

In [18]:
# Kayıp fonksiyonu ve optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Eğitim parametreleri
num_epochs = 100
for epoch in range(num_epochs):
    total_loss = 0.0
    correct = 0
    total = 0
    model.train()
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = inputs.view(inputs.size(0), 313, 29)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

print("Eğitim tamamlandı!")
torch.save(model.state_dict(), "speaker_identification_lstm.pth")
print("LSTM Modeli kaydedildi!")

Epoch [1/100], Loss: 3.7982, Accuracy: 35.51%
Epoch [2/100], Loss: 3.3208, Accuracy: 38.41%
Epoch [3/100], Loss: 2.9404, Accuracy: 38.41%
Epoch [4/100], Loss: 2.8107, Accuracy: 38.41%
Epoch [5/100], Loss: 2.7686, Accuracy: 38.41%
Epoch [6/100], Loss: 2.7219, Accuracy: 38.41%
Epoch [7/100], Loss: 2.6882, Accuracy: 38.41%
Epoch [8/100], Loss: 2.6601, Accuracy: 38.41%
Epoch [9/100], Loss: 2.6461, Accuracy: 38.41%
Epoch [10/100], Loss: 2.6464, Accuracy: 38.41%
Epoch [11/100], Loss: 2.6302, Accuracy: 38.41%
Epoch [12/100], Loss: 2.6263, Accuracy: 38.41%
Epoch [13/100], Loss: 2.6072, Accuracy: 38.41%
Epoch [14/100], Loss: 2.6629, Accuracy: 38.41%
Epoch [15/100], Loss: 2.6100, Accuracy: 38.41%
Epoch [16/100], Loss: 2.6451, Accuracy: 38.41%
Epoch [17/100], Loss: 2.6027, Accuracy: 38.41%
Epoch [18/100], Loss: 2.5968, Accuracy: 38.41%
Epoch [19/100], Loss: 2.5794, Accuracy: 38.41%
Epoch [20/100], Loss: 2.5805, Accuracy: 38.41%
Epoch [21/100], Loss: 2.5448, Accuracy: 38.41%
Epoch [22/100], Loss: 